# 02_06 Positions

*📖 Book: section 2.8*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → [numbers] → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **numbers**
>
> Project files this connects to:
>
> - `src/homer_ai/model.py`
>
> **This notebook:** The embedding of `α` is the same wherever `α` appears. How does the model know WHERE each character is?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- Why the same row for `α` everywhere hides the order of the letters
- What a positional embedding is, and how it's added to the token embedding
- That our `model.py` has no positional embedding yet

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **position** | where a token sits in the chunk: 0 for the first, 1 for the second, and so on | in `ἄνδρα`, `ρ` is at position 3 |
| **positional embedding** | a second embedding table with one row per position; its row is added to the token's row | row 3 for the 4th letter |
| **context length** | how many tokens the model reads at once: the number of positions | 24 in our training |

## 3. The idea

### ❓ What is the problem?

In [02_05](02_05_token_embeddings.ipynb) each token gets **one** row, wherever it appears. So `ἄνδρα` ("man") and the scrambled `ρνδἄα` use exactly the same five rows, just in a different order:

```
ἄνδρα   IDs [58, 38, 29, 42, 26]     rows: ἄ ν δ ρ α
ρνδἄα   IDs [42, 38, 29, 58, 26]     rows: ρ ν δ ἄ α   (same five rows)
```

👉 **Predict first:** if we add up the five rows of each word, will the two totals be the same?

In [ ]:
import torch
from torch import nn

from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

tokenizer = CharTokenizer(load_books(BOOKS_DIR)[0])
torch.manual_seed(0)
table = nn.Embedding(tokenizer.vocab_size, 32)

word = torch.tensor(tokenizer.encode("ἄνδρα"))
scrambled = torch.tensor(tokenizer.encode("ρνδἄα"))
print(word.tolist(), scrambled.tolist())
total_word = table(word).sum(dim=0)
total_scrambled = table(scrambled).sum(dim=0)
print("same total?", torch.allclose(total_word, total_scrambled))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`True`. Nothing in the rows themselves says which letter came first.

</details>

### ❓ Surely the model can still see the order?

Not really. The part of the model that reads the earlier letters is **attention** (chapter 3). It mixes in the rows of the letters before, but it only sees *which rows* are there, not *where* they sit. So the model couldn't tell `ἄνδρα` from `νἄδρα` by their earlier letters. We can only test that once we have attention, and [03_03](../03_attention/03_03_hiding_the_future.ipynb) runs the test on our real `HomerGreekLLM`.

### ❓ What is the fix?

Give every position its own row too, in a second table called the **positional embedding**. Row 0 is for the 1st place, row 1 for the 2nd, and so on. Then the model adds the two rows together, number by number:

```
position        0      1      2      3      4
token row       ἄ      ν      δ      ρ      α
+ position row  p0     p1     p2     p3     p4
= model input   ἄ+p0   ν+p1   δ+p2   ρ+p3   α+p4
```

Now `ρ` at position 3 and `ρ` at position 0 give different inputs. Like the token table, the position table starts out random and is learned in training.

Here's a tiny example with 3 numbers per row, so you can check the sums by eye (the rows are small hand-made numbers):

👉 **Predict first:** the token rows are `[1, 0, 2]` for `ν` and `[0, 3, 1]` for `ο`; the position rows are `[10, 10, 10]` and `[20, 20, 20]`. What is `νο`'s input? And `ον`'s?

In [ ]:
token_rows = {"ν": torch.tensor([1, 0, 2]), "ο": torch.tensor([0, 3, 1])}
position_rows = torch.tensor([[10, 10, 10], [20, 20, 20]])

for text in ["νο", "ον"]:
    tokens = torch.stack([token_rows[ch] for ch in text])
    print(text, "→", (tokens + position_rows).tolist())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`νο` gives `[[11, 10, 12], [20, 23, 21]]` and `ον` gives `[[10, 13, 11], [21, 20, 22]]`. Same two rows, but now the inputs differ. The order is in the numbers.

</details>

### ❓ How big is the position table?

One row per position, so as many rows as the **context length**, which is how many tokens the model reads at once. Our training chunk is 24 characters (see [02_04](02_04_sliding_window.ipynb)), so that's 24 rows, with the same 32 columns as the token table.

👉 **Predict first:** how many weights is that?

In [ ]:
context_length = 24
positions = nn.Embedding(context_length, 32)
print("shape:  ", tuple(positions.weight.shape))
print("weights:", positions.weight.numel())
grown = 11840 + positions.weight.numel()
print(f"our model's weights would grow from 11,840 to {grown:,}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

A `(24, 32)` table with 768 weights. That's a small addition to 11,840.

</details>

### ❓ Does our `model.py` have positions?

No. Here are the real lines of `HomerGreekLLM` that touch the embedding, printed straight from the code:

👉 **Predict first:** how many lines will mention a position table?

In [ ]:
import inspect

from homer_ai.model import HomerGreekLLM

for line in inspect.getsource(HomerGreekLLM).splitlines():
    if "embedding" in line:
        print(line.strip())

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Only the token embedding: `self.embedding = nn.Embedding(vocab_size, embed_dim)` and `x = self.embedding(idx)`. Nothing about positions. That's one reason our model is limited: it knows which letters came before, but not in what order. The GPT of chapter 4 does have a position table (`position_embedding` in `src/homer_ai/gpt.py`, built in [04_05](../04_building_the_model/04_05_the_full_model.ipynb)). We don't change `model.py` here.

</details>

### ❓ Can I see token + position = input?

Yes. Here are three stacked pictures for `ἄνδρα`: the five token rows, the five position rows, and their sum. These are untrained random tables, so it's the shape of the idea that counts. The last picture isn't the same as the first one, because each row has been shifted by its position.

👉 **Predict first:** will the bottom picture look more like the top one or the middle one?

In [ ]:
import matplotlib.pyplot as plt

from homer_ai.studio_style import GOOD_BAD, INK, MUTED, studio_figure

torch.manual_seed(0)
token_table = nn.Embedding(tokenizer.vocab_size, 32)
position_table = nn.Embedding(context_length, 32)

ids = torch.tensor(tokenizer.encode("ἄνδρα"))
token_part = token_table(ids).detach()
position_part = position_table(torch.arange(len(ids))).detach()
model_input = token_part + position_part

fig = studio_figure("Token + position = input", "ἄνδρα as numbers", size=(9, 7.5))
limit = max(t.abs().max().item() for t in (token_part, position_part, model_input))
panels = [
    ("token rows", token_part),
    ("+ position rows", position_part),
    ("= model input", model_input),
]
for k, (name, grid) in enumerate(panels):
    ax = fig.add_axes((0.13, 0.60 - k * 0.27, 0.8, 0.19))
    ax.set_facecolor("none")
    ax.imshow(grid, cmap=GOOD_BAD, vmin=-limit, vmax=limit, aspect="auto")
    labels = [f"{ch}  pos {p}" for p, ch in enumerate("ἄνδρα")]
    ax.set_yticks(range(5), labels, fontsize=9, color=INK)
    ax.set_xticks([])
    ax.tick_params(length=0)
    for side in ax.spines.values():
        side.set_visible(False)
    ax.set_title(name, loc="left", color=MUTED, fontsize=10)
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The bottom picture is a blend of the two above it. Each row is its letter's pattern plus its position's pattern, so `ἄ` at position 0 and `ἄ` at position 3 would look different.

</details>

### ❓ Are there other ways to give positions?

Yes. What we described is **absolute** positions: each place (0, 1, 2 …) has its own learned row. Relative positions tell the model how far apart two letters are ("3 places before"), wherever they sit in the chunk. GPT-2 uses the absolute kind, and so does the GPT we build in chapter 4. Many newer models use a relative kind. We stay with absolute positions: one extra table, and nothing else changes.

## 4. Recap

- One row per token hides the order: `ἄνδρα` and `ρνδἄα` use the same five rows (03_03 shows our model really can't tell).
- A positional embedding is a second learned table, one row per position (24 × 32 for us), added to the token row.
- Our `model.py` has no positional embedding. The GPT of chapter 4 (`gpt.py`) has one.

**You can now:**

- say why token rows alone can't tell `ἄνδρα` from `ρνδἄα`
- add a positional row to each token row

**Next:** [chapter 3: Attention](../03_attention/03_00_chapter_overview.ipynb): how each letter looks back at the letters before it.